In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

late = pd.read_csv("../data/olist_simulated_experiment.csv")
print("Loaded shape:", late.shape)
print(late[["delivery_delay_days", "treatment", "sim_negative_review"]].head())

Loaded shape: (7661, 28)
   delivery_delay_days  treatment  sim_negative_review
0            11.933171          0                    1
1             9.919375          0                    1
2             0.041262          0                    0
3             7.791238          1                    1
4             1.561644          1                    1


In [2]:
Y = late["sim_negative_review"]
X = late["delivery_delay_days"]

theta = np.cov(Y, X)[0, 1] / np.var(X)
print(f"Theta: {theta:.6f}")

X_mean = X.mean()
late["Y_cuped"] = Y - theta * (X - X_mean)

print(f"\nOriginal outcome variance: {Y.var():.6f}")
print(f"CUPED-adjusted outcome variance: {late['Y_cuped'].var():.6f}")
print(f"Variance reduction: {(1 - late['Y_cuped'].var()/Y.var())*100:.1f}%")

Theta: 0.011703

Original outcome variance: 0.244679
CUPED-adjusted outcome variance: 0.218707
Variance reduction: 10.6%


In [3]:
from scipy import stats

# Original (unadjusted) comparison, using t-test on means for a fair comparison basis
control_orig = late[late["treatment"]==0]["sim_negative_review"]
treated_orig = late[late["treatment"]==1]["sim_negative_review"]
t_orig, p_orig = stats.ttest_ind(treated_orig, control_orig)
diff_orig = treated_orig.mean() - control_orig.mean()
se_orig = np.sqrt(treated_orig.var()/len(treated_orig) + control_orig.var()/len(control_orig))
ci_orig = (diff_orig - 1.96*se_orig, diff_orig + 1.96*se_orig)

# CUPED-adjusted comparison
control_cuped = late[late["treatment"]==0]["Y_cuped"]
treated_cuped = late[late["treatment"]==1]["Y_cuped"]
t_cuped, p_cuped = stats.ttest_ind(treated_cuped, control_cuped)
diff_cuped = treated_cuped.mean() - control_cuped.mean()
se_cuped = np.sqrt(treated_cuped.var()/len(treated_cuped) + control_cuped.var()/len(control_cuped))
ci_cuped = (diff_cuped - 1.96*se_cuped, diff_cuped + 1.96*se_cuped)

print("=== Without CUPED ===")
print(f"Effect: {diff_orig*100:.2f}pts, 95% CI: [{ci_orig[0]*100:.2f}, {ci_orig[1]*100:.2f}], width: {(ci_orig[1]-ci_orig[0])*100:.2f}pts")
print(f"p-value: {p_orig:.2e}")
print()
print("=== With CUPED ===")
print(f"Effect: {diff_cuped*100:.2f}pts, 95% CI: [{ci_cuped[0]*100:.2f}, {ci_cuped[1]*100:.2f}], width: {(ci_cuped[1]-ci_cuped[0])*100:.2f}pts")
print(f"p-value: {p_cuped:.2e}")
print()
ci_width_reduction = (1 - (ci_cuped[1]-ci_cuped[0])/(ci_orig[1]-ci_orig[0])) * 100
print(f"CI width reduction from CUPED: {ci_width_reduction:.1f}%")

=== Without CUPED ===
Effect: -8.56pts, 95% CI: [-10.77, -6.35], width: 4.41pts
p-value: 3.31e-14

=== With CUPED ===
Effect: -8.82pts, 95% CI: [-10.90, -6.73], width: 4.17pts
p-value: 1.38e-16

CI width reduction from CUPED: 5.5%


## Summary

This notebook applies CUPED (Controlled-experiment Using Pre-Experiment Data) variance reduction
to the simulated experiment, using delivery delay severity as the covariate.

**Why delay severity is a valid CUPED covariate:**
- Known before the treatment decision is made (the order is already late)
- Strongly correlated with the outcome (validated in Notebook 1/3: 15% to 80%+ negative review
  rate across delay-severity buckets)
- Not affected by treatment -- sending a notification doesn't change how late a package is

**Results:**
- Theta (regression coefficient of outcome on delay days): 0.0117
- Outcome variance reduced by **10.6%**
- 95% CI narrowed from 4.41pts to 4.17pts wide (a **5.5%** reduction), consistent with theory
  since CI width scales with the square root of variance, not variance directly
- Point estimate essentially unchanged (-8.56pts vs -8.82pts) -- confirming CUPED adjusts
  precision, not the effect estimate itself

**Takeaway:** a modest but genuine precision gain, achieved with zero additional data
collection, since delay severity was already being measured as part of the eligibility criteria
for this experiment.